Importamos las librerias de Spark

In [ ]:
!pip install pyspark

In [ ]:
%restart_python

Paso 1. Cargar y asignar nombres a las columnas

Creamos la sesion

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, to_date

spark = SparkSession.builder.appName("Villarica").getOrCreate()

Cargamos el archivo de datos csv de Villarica

In [ ]:
df = spark.read.csv("/Volumes/workspace/bd_villarica/grupo_01/CODIGO/RESULTADOS-Villarica.csv",header=False,sep=";",inferSchema=True,)

Damos Nombres a lasa columnas

In [ ]:
df = df.toDF(
    "id_fact_venta",
    "id_venta",
    "fecha",
    "producto",
    "categoria",
    "cantidad",
    "precio",
    "metodo_pago"
)

Paso 2. Revisar la estructura de los datos

In [ ]:
# Mostrar tipos de datos
df.printSchema()

# Mostrar estadísticas generales
df.describe().show()

# Contar registros
print("Total de registros:", df.count())

Paso 3. Eliminar espacios innecesarios

In [ ]:
# Limpiar espacios en columnas de texto
df = df.withColumn("producto", trim(col("producto"))).withColumn("categoria", trim(col("categoria"))).withColumn("metodo_pago", trim(col("metodo_pago")))

df.show(5, truncate=False)

Paso 4. Identificar y tratar valores nulos

In [ ]:
# Revisar valores nulos
df.select([col(c).isNull().cast("int").alias(c)
    for c in df.columns
]).groupBy().sum().show()

df = df.na.drop(
    subset=[
        "id_fact_venta",
        "id_venta",
        "fecha",
        "producto",
        "categoria",
        "cantidad",
        "precio",
        "metodo_pago"
    ]
)

Paso 5. Corregir tipos de datos

In [ ]:
df = df.withColumn("id_fact_venta", col("id_fact_venta").cast("integer")).withColumn("id_venta", col("id_venta").cast("integer")).withColumn("fecha", to_date(col("fecha"), "yyyy-MM-dd")).withColumn("cantidad", col("cantidad").cast("integer")).withColumn("precio", col("precio").cast("decimal(12,2)"))

df.printSchema()

Paso 6. Filtrar registros incorrectos

In [ ]:
df = df.filter(
    (col("id_fact_venta") > 0) &
    (col("id_venta") > 0) &
    (col("fecha").isNotNull()) &
    (col("producto") != "") &
    (col("categoria") != "") &
    (col("cantidad") > 0) &
    (col("precio") > 0) &
    (col("metodo_pago") != "")
)

df.show(10, truncate=False)

Paso 7. Eliminar registros duplicados

In [ ]:
df = df.dropDuplicates()

print("Registros después de eliminar duplicados:", df.count())

Paso 8. Crear una nueva columna calculada

In [ ]:
df_total = df = df.withColumn(
    "importe_total",
    col("precio")
)

In [ ]:
from pyspark.sql.functions import sum as spark_sum

df.select(
    spark_sum("cantidad").alias("Total_Unidades"),
    spark_sum("importe_total").alias("Total_Ventas")
).show()


Paso 9. Seleccionar columnas necesarias

In [ ]:
# Seleccionar información para el análisis
df = df.select(
    "id_fact_venta",
    "id_venta",
    "fecha",
    "producto",
    "categoria",
    "cantidad",
    "precio",
    "importe_total",
    "metodo_pago"
)

df.show(10, truncate=False)

Paso 10. Agrupar y analizar las ventas

Agrupar las ventas por categoría:

In [ ]:
df_ventas_categoria=df.groupBy("categoria").sum("cantidad", "importe_total")
display(df_ventas_categoria)

Agrupar las ventas por método de pago:

In [ ]:
df_ventas_pago=df.groupBy("metodo_pago").sum("importe_total")
display(df_ventas_pago)

Filtrar productos de la categoría Panes:

In [ ]:
df_panes_c=df.filter(
    col("categoria") == "Panes"
).select(
    "producto",
    "cantidad",
    "precio",
    "importe_total"
)
display(df_panes_c)

Analizar las ventas por producto:

In [ ]:
df.groupBy("producto") \
    .sum("cantidad", "importe_total") \
    .orderBy(col("sum(importe_total)").desc()) \
    .show(10, truncate=False)

CREACION BASE DE DATOS

In [ ]:
%sql

CREATE DATABASE IF NOT EXISTS bd_villarica;


In [ ]:
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("bd_villarica.tb_ventas_villarica")

In [ ]:

dfVentasGuardadas = spark.table('bd_villarica.tb_ventas_villarica')

display(dfVentasGuardadas)


GRAFICOS

In [ ]:
dfVentasCategoria = df.groupBy("categoria").sum("importe_total")
display(dfVentasCategoria)

In [ ]:

dfProductosVendidos = df.groupBy("producto").sum("cantidad")

dfProductosVendidos = dfProductosVendidos.orderBy(
    col("sum(cantidad)").desc()
).limit(10)

display(dfProductosVendidos)


In [ ]:

dfVentasPago = df.groupBy("metodo_pago").sum("importe_total")

display(dfVentasPago)


EXPERIMENTO 1: Big Data – Evaluación del rendimiento mediante particiones en Apache Spark

In [ ]:
import time
from pyspark.sql.functions import col, sum as spark_sum

# Escenarios de particionamiento
particiones = [1, 2, 4]

resultados_particiones = []

for numero in particiones:

    inicio = time.perf_counter()

    # Distribuir los datos en diferentes particiones
    df_particionado = df.repartition(numero)

    # Ejecutar agrupación y cálculo de ventas
    resultado = df_particionado.groupBy("categoria").agg(
        spark_sum("cantidad").alias("Total_Unidades"),
        spark_sum(
            col("cantidad") * col("precio")
        ).alias("Total_Ventas")
    )

    # Ejecutar el procesamiento
    resultado.collect()

    fin = time.perf_counter()

    resultados_particiones.append((
        numero,
        round(fin - inicio, 3)
    ))

# Crear tabla comparativa
df_particiones = spark.createDataFrame(
    resultados_particiones,
    ["Numero_Particiones", "Tiempo_Segundos"]
)

display(df_particiones)

EXPERIMENTO 1 (bloque nuevo): Tiempo de procesamiento según el volumen de datos

In [ ]:

import time
from functools import reduce

# Dataset base
df_base = df

# Volúmenes de prueba
multiplicadores = [1, 2, 5, 10]

resultados = []

for factor in multiplicadores:

    # Generar volumen de prueba mediante UNION
    data_prueba = reduce(
        lambda a, b: a.union(b),
        [df_base] * factor
    )

    inicio = time.perf_counter()

    # Ejecutar procesamiento
    total_registros = data_prueba.count()

    fin = time.perf_counter()

    tiempo = fin - inicio

    resultados.append((
        factor,
        total_registros,
        round(tiempo, 3)
    ))

# Mostrar resultados
columnas = [
    "Multiplicador",
    "Total_Registros",
    "Tiempo_Segundos"
]

df_resultados = spark.createDataFrame(
    resultados, columnas
)

display(df_resultados)


EXPERIMENTO 2: BI – Análisis de la evolución de ventas por día

In [ ]:

from pyspark.sql.functions import col, sum as spark_sum

# Corregir el importe total
df_bi = df.withColumn(
    "importe_total",
    col("cantidad") * col("precio")
)

# 1. Ventas por categoría
ventas_categoria = df_bi.groupBy("categoria").agg(
    spark_sum("cantidad").alias("Total_Unidades"),
    spark_sum("importe_total").alias("Total_Ventas")
).orderBy(col("Total_Ventas").desc())

display(ventas_categoria)

# 2. Ventas por método de pago
ventas_pago = df_bi.groupBy("metodo_pago").agg(
    spark_sum("importe_total").alias("Total_Ventas")
).orderBy(col("Total_Ventas").desc())

display(ventas_pago)

# 3. Productos más vendidos
productos_vendidos = df_bi.groupBy("producto").agg(
    spark_sum("cantidad").alias("Total_Unidades"),
    spark_sum("importe_total").alias("Total_Ventas")
).orderBy(col("Total_Unidades").desc()).limit(10)

display(productos_vendidos)


In [ ]:

from pyspark.sql.functions import col, sum as spark_sum

# Calcular importe total
df_evolucion = df.withColumn(
    "importe_total",
    col("cantidad") * col("precio")
)

# Agrupar las ventas por fecha
ventas_diarias = df_evolucion.groupBy("fecha").agg(
    spark_sum("importe_total").alias("Total_Ventas"),
    spark_sum("cantidad").alias("Total_Unidades")
).orderBy("fecha")

# Mostrar resultados
display(ventas_diarias)
